# 火灾疏散项目 — Colab 训练脚本
## 使用方法
1. 菜单栏 → **修改** → **笔记本设置** → 硬件加速器选 **T4 GPU**
2. 依次点每个单元格左边的 **播放按钮**，从上到下逐个运行
3. 训练完模型自动存到你的 Google Drive，回去下载到本地

In [ ]:
# ============================================================
# 第 1 步：挂载 Google Drive + 安装依赖
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/huo-zai-checkpoints'
os.makedirs(DRIVE_PATH, exist_ok=True)

In [ ]:
# ============================================================
# 第 2 步：克隆你的代码 + 安装依赖
# ============================================================
!git clone https://github.com/confidentismylife/LonlyStydue.git
%cd LonlyStydue
!pip install -q torch numpy pandas matplotlib scipy scikit-learn tqdm
print('依赖安装完成')

In [ ]:
# ============================================================
# 第 3 步：下载 ETH/UCY 数据集（约 500MB，只需下载一次）
# ============================================================
!mkdir -p datasets && cd datasets
!wget -q https://github.com/crowdbotp/OpenTraj/raw/master/datasets/ETH_UCY/ethucy.zip
!unzip -q ethucy.zip
!rm ethucy.zip
%cd /content/LonlyStydue

# 验证数据下载成功
import os
txt_count = sum(1 for root, dirs, files in os.walk('datasets') for f in files if f.endswith('.txt'))
print(f'找到 {txt_count} 个 .txt 数据文件')

In [ ]:
# ============================================================
# 第 4 步：检查 GPU
# ============================================================
import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')
    !nvidia-smi --query-gpu=memory.total --format=csv,noheader

In [ ]:
# ============================================================
# 第 5 步：修改数据路径指向 Colab 本地，然后开始训练
# ============================================================
import sys
sys.path.insert(0, '.')

import src.config as cfg
# 把数据路径指向刚下载的数据集
cfg.ETH_PATH = os.path.join('/content/LonlyStydue/datasets', 'eth')
cfg.UCY_PATH = os.path.join('/content/LonlyStydue/datasets', 'ucy')

# 建议的超参数（T4 完全够用）
cfg.EPOCHS = 100
cfg.BATCH_SIZE = 128
cfg.LR = 0.001

# 临时修改 train.py 里的数据路径查找逻辑
# 直接指定用 Colab 的数据目录
social_stgcnn_data = '/content/LonlyStydue/datasets'

# 加载数据
import numpy as np
from src.data_loader import load_eth_ucy_file, extract_trajectories, normalize_trajectories

all_files = []
for root, dirs, files in os.walk(social_stgcnn_data):
    for f in files:
        if f.endswith('.txt'):
            all_files.append(os.path.join(root, f))

print(f'找到 {len(all_files)} 个数据文件')

all_obs, all_preds = [], []
for fpath in all_files:
    raw = load_eth_ucy_file(fpath)
    obs, preds = extract_trajectories(raw)
    if len(obs) > 0:
        all_obs.append(obs)
        all_preds.append(preds)

all_obs = np.concatenate(all_obs)
all_preds = np.concatenate(all_preds)
print(f'总轨迹数: {len(all_obs)}')

# 归一化
obs_norm, preds_norm = normalize_trajectories(all_obs, all_preds)

# 8:2 划分训练/验证
split = int(len(obs_norm) * 0.8)
train_obs, train_preds = obs_norm[:split], preds_norm[:split]
val_obs, val_preds = obs_norm[split:], preds_norm[split:]
print(f'训练集: {len(train_obs)} 条 | 验证集: {len(val_obs)} 条')

In [ ]:
# ============================================================
# 第 6 步：创建 DataLoader + 模型 + 训练
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from src.model import SimpleLSTM
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

# Dataset
class TrajDataset(Dataset):
    def __init__(self, obs, preds):
        self.obs = torch.from_numpy(obs).float()
        self.preds = torch.from_numpy(preds).float()
    def __len__(self):
        return len(self.obs)
    def __getitem__(self, idx):
        return self.obs[idx], self.preds[idx]

train_loader = DataLoader(TrajDataset(train_obs, train_preds), batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TrajDataset(val_obs, val_preds), batch_size=cfg.BATCH_SIZE, shuffle=False)

# 模型
model = SimpleLSTM(hidden_dim=64, pred_len=cfg.PRED_LEN).to(device)
print(f'模型参数量: {sum(p.numel() for p in model.parameters()):,}')

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

# 训练循环
best_ade = float('inf')
for epoch in range(1, cfg.EPOCHS + 1):
    t0 = time.time()

    # Train
    model.train()
    train_loss = 0.0
    for obs, preds in train_loader:
        obs, preds = obs.to(device), preds.to(device)
        optimizer.zero_grad()
        loss = criterion(model(obs), preds)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * obs.size(0)
    train_loss /= len(train_loader.dataset)

    # Validate
    model.eval()
    ade_sum, fde_sum, count = 0.0, 0.0, 0
    with torch.no_grad():
        for obs, preds in val_loader:
            obs, preds = obs.to(device), preds.to(device)
            output = model(obs)
            diff = output - preds
            ade_sum += torch.norm(diff, dim=2).mean(dim=1).sum().item()
            fde_sum += torch.norm(diff[:, -1, :], dim=1).sum().item()
            count += obs.size(0)
    val_ade = ade_sum / count
    val_fde = fde_sum / count
    scheduler.step(val_ade)

    # 保存最佳模型
    if val_ade < best_ade:
        best_ade = val_ade
        os.makedirs('checkpoints', exist_ok=True)
        torch.save(model.state_dict(), 'checkpoints/best_model.pth')

    if epoch <= 5 or epoch % 20 == 0:
        print(f'Epoch {epoch:3d}/{cfg.EPOCHS} | Loss: {train_loss:.4f} | '
              f'ADE: {val_ade:.3f}m | FDE: {val_fde:.3f}m | {time.time()-t0:.1f}s')

print(f'\n训练完成！最佳 ADE: {best_ade:.3f} 米')

In [ ]:
# ============================================================
# 第 7 步：保存模型到 Google Drive
# ============================================================
import shutil
if os.path.exists('checkpoints/best_model.pth'):
    dest = os.path.join(DRIVE_PATH, 'best_model.pth')
    shutil.copy('checkpoints/best_model.pth', dest)
    print(f'模型已保存到: {dest}')
    print('Colab 断开后模型不会丢失，在你的 Google Drive 里能找到')
else:
    print('未找到模型文件')

In [ ]:
# ============================================================
# 第 8 步：随机抽 3 条轨迹画图看看效果
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

model.eval()
sample_ids = np.random.choice(len(val_obs), 3, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, idx in enumerate(sample_ids):
    obs_i = torch.from_numpy(val_obs[idx]).float().unsqueeze(0).to(device)
    gt_i = val_preds[idx]
    with torch.no_grad():
        pred_i = model(obs_i).squeeze(0).cpu().numpy()
    obs_i = obs_i.squeeze(0).cpu().numpy()

    ax = axes[i]
    ax.plot(obs_i[:, 0], obs_i[:, 1], 'b-o', label='Observed')
    ax.plot(pred_i[:, 0], pred_i[:, 1], 'r--o', label='Predicted')
    ax.plot(gt_i[:, 0], gt_i[:, 1], 'g-o', alpha=0.5, label='Ground Truth')
    ax.set_title(f'Sample {i+1}')
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('colab_results.png', dpi=100)
plt.show()
print(f'对比：匀速基线 ADE=0.48m — 看你的 LSTM 有没有超过它')